**Deep CNN for Fashion MNIST Classification using PyTorch**
This project implements a custom deep convolutional neural network (CNN) in PyTorch for classifying images from the Fashion MNIST (FMNIST) dataset. The FMNIST dataset consists of 28×28 grayscale images representing 10 categories of clothing items, offering a modern alternative to the MNIST digit classification benchmark.

Model Architecture:
The model begins with three convolutional layers using Conv2d, each followed by Layer Normalization, ReLU activation, and max pooling.

Intermediate dropout layers are included to improve generalization and reduce overfitting.

After the final convolutional block, the output is flattened and passed through two fully connected (FC) layers:

Each FC layer is normalized using LayerNorm, and dropout is again applied.

A custom bias tensor (a learned linear scaling vector) is added after each FC output, introducing position-specific learning dynamics.

The final layer is a 10-class output layer suitable for multi-class classification.

Training Setup:
Loss Function: Cross-Entropy Loss, ideal for classification tasks

Optimizer: Adam optimizer with L2 regularization (weight_decay=0.01) to prevent overfitting

Weight Initialization: Custom Kaiming He initialization is used for all convolutional and linear layers, ensuring good gradient flow at the start of training

Key Innovations:
Integration of a custom bias tensor with the FC outputs introduces novel positional modulation

Careful architectural choices, including multiple dropout and normalization layers, provide robustness and stability

Dynamic input size computation via a dummy forward pass ensures flexibility across different input shapes

This project demonstrates a strong understanding of modern deep learning practices in PyTorch, emphasizing modularity, normalization strategies, and robust architectural design for image classification tasks.

In [ ]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

/kaggle/input/fashionmnist/t10k-labels-idx1-ubyte
/kaggle/input/fashionmnist/t10k-images-idx3-ubyte
/kaggle/input/fashionmnist/fashion-mnist_test.csv
/kaggle/input/fashionmnist/fashion-mnist_train.csv
/kaggle/input/fashionmnist/train-labels-idx1-ubyte
/kaggle/input/fashionmnist/train-images-idx3-ubyte
/kaggle/input/model_weights/pytorch/default/1/model_weights after gpu 250 epoch.pth


In [20]:
mnist_train = pd.read_csv("/kaggle/input/fashionmnist/fashion-mnist_train.csv")
mnist_test = pd.read_csv("/kaggle/input/fashionmnist/fashion-mnist_test.csv")

In [21]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [22]:

print(mnist_train.columns)

y_train=mnist_train['label']
y_test=mnist_test['label']
mnist_train=mnist_train.drop('label',axis=1)
mnist_test=mnist_test.drop('label',axis=1)

Index(['label', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5', 'pixel6',
       'pixel7', 'pixel8', 'pixel9',
       ...
       'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779', 'pixel780',
       'pixel781', 'pixel782', 'pixel783', 'pixel784'],
      dtype='object', length=785)


In [23]:
print(y_test.value_counts())

label
0    1000
1    1000
2    1000
3    1000
8    1000
6    1000
5    1000
4    1000
7    1000
9    1000
Name: count, dtype: int64


In [24]:
mnist_train=mnist_train.to_numpy()
mnist_test=mnist_test.to_numpy() 
print(mnist_train.shape)

(60000, 784)


In [25]:
import numpy as np
from sklearn.preprocessing import StandardScaler


# Create a StandardScaler instance
scaler = StandardScaler()

# Fit and transform the data
mnist_train = scaler.fit_transform(mnist_train)
mnist_test=scaler.transform(mnist_test)

mnist_train = mnist_train.reshape(-1,28,28) 
mnist_test=mnist_test.reshape(-1,28,28)

In [26]:
import torch

X_train_tensor=torch.tensor(mnist_train,dtype=torch.float32).to(device)
print(X_train_tensor.shape)
X_train_tensor=torch.unsqueeze(X_train_tensor,1)
print(X_train_tensor.shape)

X_test_tensor=torch.tensor(mnist_test,dtype=torch.float32).to(device)
X_test_tensor=torch.unsqueeze(X_test_tensor,1)

y_train_tensor=torch.tensor(y_train,dtype=torch.int64).to(device)

y_test_tensor=torch.tensor(y_test,dtype=torch.int64).to(device)

torch.Size([60000, 28, 28])
torch.Size([60000, 1, 28, 28])


In [27]:
print(X_test_tensor.shape)

torch.Size([10000, 1, 28, 28])


In [28]:
print(X_train_tensor.shape)
print(y_train_tensor)

torch.Size([60000, 1, 28, 28])
tensor([2, 9, 6,  ..., 8, 8, 7], device='cuda:0')


In [29]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

train_data=TensorDataset(X_train_tensor,y_train_tensor)
batch_size=342
train_loader=DataLoader(train_data,batch_size=batch_size, shuffle=True)


In [30]:
seed=42
torch.manual_seed(seed)  # For PyTorch on CPU
torch.cuda.manual_seed(seed)  # For PyTorch on GPU

In [31]:
class ClassificationModel(nn.Module):
    def __init__(self):
        super(ClassificationModel,self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1,out_channels=128,kernel_size=3)
        self.bn1=nn.LayerNorm([128,26,26])

        self.maxpool1=nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(in_channels=128,out_channels=100,kernel_size=3)
        self.bn2=nn.LayerNorm([100,11,11])
        self.conv3=nn.Conv2d(in_channels=100,out_channels=72,kernel_size=3)
        self.bn3=nn.LayerNorm([72,3,3])
        self.dropout3 = nn.Dropout(0.3)
        self.relu = nn.ReLU()     
        self.calsize((1,28,28))
        self.fc1=nn.Linear(648,78)
        self.ln1=nn.LayerNorm(78)
        self.dropout4=nn.Dropout(0.3)
        self.fc2=nn.Linear(78,78)
        self.ln2=nn.LayerNorm(78)
        self.dropout5=nn.Dropout(0.25)
        self.outputlayer=nn.Linear(78,10)
        self.initialize_weights()
        
    def initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    def calsize(self,inputshape):
        dummy_input = torch.zeros(1, *inputshape)
        
        # Pass it through the convolutional layers to determine the output size
        x=self.conv1(dummy_input)
        print(x.shape,"conv1")
        x = self.maxpool1(self.relu(self.bn1(x)))
        x=self.conv2(x)
        print(x.shape,'conv2')
        x = self.maxpool1(self.relu(self.bn2(x)))
        x=self.relu(self.bn3(self.conv3(x)))
        
        # Calculate the flattened size for the fully connected layer
        self.linear_input_size = x.numel()

    def forward(self,inputx):
        x = self.maxpool1(self.relu(self.bn1(self.conv1(inputx))))
        x = self.maxpool1(self.relu(self.bn2(self.conv2(x))))
        x = self.dropout3(self.relu(self.bn3(self.conv3(x))))
        conv3outflattened=torch.flatten(x,start_dim=1)

        fc1output=self.fc1(conv3outflattened)

        tensor1=(torch.arange(1,79))/10
        tensor1=tensor1.to(device)
        fc1output=fc1output + tensor1
        
        fc1output=self.dropout4(self.ln1(fc1output))

        fc2output=self.fc2(fc1output)
        fc2output=fc2output + fc1output
        fc2output=fc2output + tensor1
        fc2output=self.dropout5(self.ln2(fc2output))
        out=self.outputlayer(fc2output)
        return out

In [32]:
model= ClassificationModel().to(device)
criterion = nn.CrossEntropyLoss()  # Cross-entropy loss for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001,weight_decay=0.01)
scheduled=0


torch.Size([1, 128, 26, 26]) conv1
torch.Size([1, 100, 11, 11]) conv2
648 here


In [33]:
def freeze_layer(layer, freeze=True):
    for param in layer.parameters():
        param.requires_grad = not freeze  # If freeze=True, set requires_grad=False

In [34]:
for epoch in range(1,150):
    epoch_loss=0
    if epoch ==1:
        freeze_layer(model.conv3)
        freeze_layer(model.fc2)
        optimizer=optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
    if epoch==50:
        freeze_layer(model.conv3,freeze=False)
        optimizer=optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

    if epoch==85:
        freeze_layer(model.fc2,freeze=False)
        optimizer=optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0007)

    if epoch==110:
        optimizer=optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

    if epoch==125:
        optimizer=optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.00001) 
 
    if epoch==135:
        optimizer=optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.000001) 
  
        
        
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        inputs,targets=inputs.to(device),targets.to(device)
        output=model(inputs)
        loss=criterion(output,targets)
        loss.backward()
        optimizer.step()
        if scheduled:
           scheduler.step()
        epoch_loss+=loss.item()
        
    print(f'loss for epoch {epoch}:{epoch_loss/len(train_loader)}')

loss for epoch 1:0.9362091168083928
loss for epoch 2:0.5111155684360049
loss for epoch 3:0.43259772031821986
loss for epoch 4:0.38920603557066485
loss for epoch 5:0.3566949472508647
loss for epoch 6:0.33628494411029597
loss for epoch 7:0.3181309560313821
loss for epoch 8:0.30199660056016664
loss for epoch 9:0.2884160586717454
loss for epoch 10:0.27262328302657063
loss for epoch 11:0.2620618230747906
loss for epoch 12:0.2515925612639297
loss for epoch 13:0.23850873180411078
loss for epoch 14:0.2284809642217376
loss for epoch 15:0.22432736535979944
loss for epoch 16:0.21673088131303136
loss for epoch 17:0.21272123579613186
loss for epoch 18:0.20099252500486645
loss for epoch 19:0.1970994667234746
loss for epoch 20:0.1882126361385665
loss for epoch 21:0.18281291611492634
loss for epoch 22:0.17500408108092166
loss for epoch 23:0.17082325491884892
loss for epoch 24:0.16416107160462576
loss for epoch 25:0.15907923174514013
loss for epoch 26:0.15257734517482194
loss for epoch 27:0.15005379149

In [36]:
correct=0
with torch.no_grad():
    output=model(X_test_tensor)
    corr = torch.argmax(output, axis = 1) == torch.as_tensor(y_test_tensor)
    correct = torch.count_nonzero(corr)
print(correct)
print("accuracy",correct/10000)

tensor(9210, device='cuda:0')
accuracy tensor(0.9210, device='cuda:0')
